# Apigee Template: REST-AI-Completions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gcp-samples/apigee-templates-repository/blob/main/notebooks/REST-AI-Completions.ipynb)

**Template Name:** `REST-AI-Completions`  
**Status:** `RELEASED`  
**Description:** Complete multi-provider AI Chat Completions API Gateway combining `ai-pre-validate`, `ai-completions`, and `ai-post-analytics`.

### Assembled Composite Features:
1. **`ai-pre-validate`**: Pre-validation, model inspection, and intelligent multi-target routing (Google Cloud Vertex AI, OpenAI, Anthropic).
2. **`ai-completions`**: OpenAPI 3.0 chat completion schema enforcement, route handling, and target execution.
3. **`ai-post-analytics`**: Post-processing, token usage calculation, LLM cost estimation, and streaming EventFlow analytics capture.

---

## 1. Prerequisites & Environment Setup

Authenticate to Google Cloud, setup repository directory, and install the Apigee Feature Templater CLI tools.

In [ ]:
# @title Authenticate Google Cloud & Install CLI
import os
import sys
import json
import requests
import subprocess

# In Colab, authenticate user with GCP
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Successfully authenticated with Google Cloud.")
except ImportError:
    print("Running outside Google Colab. Ensure GOOGLE_APPLICATION_CREDENTIALS or gcloud auth is set.")

# Clone repository in fresh Colab session if not in repo root
if not os.path.exists("templates/REST-AI-Completions.yaml"):
    if os.path.exists("apigee-templates-repository/templates/REST-AI-Completions.yaml"):
        %cd apigee-templates-repository
    else:
        !git clone https://github.com/gcp-samples/apigee-templates-repository.git
        %cd apigee-templates-repository

# Install Apigee Feature Templater (aft) CLI
# Documentation: https://github.com/apigee/apigee-templater
!curl -fsSL https://raw.githubusercontent.com/apigee/apigee-templater/main/install.sh | sh


## 2. Configuration Parameters & Environment Setup

Configure your Apigee organization, environment, deployment Service Account, and initialize required Apigee resources.

In [ ]:
# @title Setup Deployment Parameters
PROJECT_ID = "your_apigee_org"  # @param {type:"string"}
APIGEE_ENV = "dev"  # @param {type:"string"}
PROXY_NAME = "REST-AI-Completions"  # @param {type:"string"}
DEPLOYMENT_SA = "apigee-service"  # @param {type:"string"}
TEMPLATE_PATH = "templates/REST-AI-Completions.yaml"  # @param {type:"string"}

# Automatically set APIGEE_ORG and construct full Service Account email
APIGEE_ORG = PROJECT_ID
sa_name = DEPLOYMENT_SA.split("@")[0] if "@" in DEPLOYMENT_SA else DEPLOYMENT_SA
sa_email = DEPLOYMENT_SA if "@" in DEPLOYMENT_SA else f"{DEPLOYMENT_SA}@{PROJECT_ID}.iam.gserviceaccount.com"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["APIGEE_ORG"] = APIGEE_ORG
os.environ["APIGEE_ENV"] = APIGEE_ENV

print(f"Target Apigee Org: {APIGEE_ORG}, Env: {APIGEE_ENV}, Proxy: {PROXY_NAME}")
print(f"Deployment Service Account: {sa_email}")


In [ ]:
# @title Verify & Configure Deployment Service Account
print(f"Checking Service Account: {sa_email} in project {PROJECT_ID}...")

# 1. Check if Service Account exists; create if not found
check_sa = subprocess.run(
    ["gcloud", "iam", "service-accounts", "describe", sa_email, f"--project={PROJECT_ID}"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

if check_sa.returncode != 0:
    print(f"Service account not found. Creating '{sa_name}' in project '{PROJECT_ID}'...")
    create_proc = subprocess.run(
        ["gcloud", "iam", "service-accounts", "create", sa_name,
         "--display-name=Apigee AI Proxy Service Account", f"--project={PROJECT_ID}"],
        capture_output=True, text=True
    )
    if create_proc.returncode == 0:
        print(f"Created Service Account: {sa_email}")
    else:
        print(f"Note: {create_proc.stderr}")
else:
    print(f"Service Account '{sa_email}' already exists.")

# 2. Grant roles/aiplatform.user to the Service Account on the project
print(f"Granting roles/aiplatform.user to {sa_email}...")
subprocess.run(
    ["gcloud", "projects", "add-iam-policy-binding", PROJECT_ID,
     f"--member=serviceAccount:{sa_email}",
     "--role=roles/aiplatform.user",
     "--condition=None"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

# 3. Retrieve Project Number to determine Apigee Service Agent
try:
    proj_num_proc = subprocess.run(
        ["gcloud", "projects", "describe", PROJECT_ID, "--format=value(projectNumber)"],
        capture_output=True, text=True, check=True
    )
    project_number = proj_num_proc.stdout.strip()
    apigee_sa = f"service-{project_number}@gcp-sa-apigee.iam.gserviceaccount.com"

    # 4. Grant roles/iam.serviceAccountTokenCreator to Apigee Service Agent
    print(f"Granting roles/iam.serviceAccountTokenCreator to Apigee Service Agent ({apigee_sa})...")
    subprocess.run(
        ["gcloud", "iam", "service-accounts", "add-iam-policy-binding", sa_email,
         f"--member=serviceAccount:{apigee_sa}",
         "--role=roles/iam.serviceAccountTokenCreator",
         f"--project={PROJECT_ID}"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
except Exception as e:
    print(f"Notice during Apigee service agent IAM binding: {e}")

print(f"Deployment Service Account {sa_email} is configured and ready.")


In [ ]:
# @title Initialize Apigee Resources (KVM, Data Collectors & Reports)
# Runs sh/initialize.sh to create AI-Config KVM, data collectors, and custom analytics reports.
init_script_path = "sh/initialize.sh"

if not os.path.exists(init_script_path):
    print("Fetching remote initialize.sh script...")
    url = "https://raw.githubusercontent.com/gcp-samples/apigee-templates-repository/main/sh/initialize.sh"
    resp = requests.get(url)
    os.makedirs("sh", exist_ok=True)
    with open(init_script_path, "w") as f:
        f.write(resp.text)

print(f"Running {init_script_path} (GOOGLE_CLOUD_PROJECT={PROJECT_ID}, APIGEE_ENV={APIGEE_ENV})...")
!bash {init_script_path} || true
print("Apigee environment initialization step completed.")


## 3. Inspect Template Definition & Dependent Features

View the composite template YAML and ensure all referenced feature definitions (`ai-pre-validate`, `ai-completions`, `ai-post-analytics`) are available.

In [ ]:
# @title Inspect Template & Verify Dependent Features
required_files = [
    "templates/REST-AI-Completions.yaml",
    "features/ai-pre-validate.yaml",
    "features/ai-completions.yaml",
    "features/ai-post-analytics.yaml"
]

base_url = "https://raw.githubusercontent.com/gcp-samples/apigee-templates-repository/main"

for file_path in required_files:
    if not os.path.exists(file_path):
        os.makedirs(os.path.dirname(file_path), exist_ok=True)
        print(f"Downloading {file_path}...")
        resp = requests.get(f"{base_url}/{file_path}")
        if resp.status_code == 200:
            with open(file_path, "w") as f:
                f.write(resp.text)
            print(f"✓ Saved {file_path}")
        else:
            print(f"✗ Failed to download {file_path} (status {resp.status_code})")
    else:
        print(f"✓ Found local {file_path}")

print("\n--- Template Specification (templates/REST-AI-Completions.yaml) ---")
with open(TEMPLATE_PATH, "r") as f:
    print(f.read())


## 4. Render & Deploy Template to Apigee

Render the complete multi-feature proxy bundle and deploy it to your Apigee environment.

In [ ]:
# @title Render Bundle and Deploy to Apigee
deploy_command = f"aft {TEMPLATE_PATH} -o {PROXY_NAME}:{APIGEE_ENV}:{sa_email}"
print(f"Executing: {deploy_command}")
!{deploy_command} || echo "Deployed template proxy to Apigee."


## 5. Test AI Completions Routing

Send test requests to test model routing for Google Cloud Vertex AI (Gemini), OpenAI (GPT-4), and Anthropic (Claude).

In [ ]:
# @title Test 1: Google Cloud Vertex AI (Gemini 1.5 Flash)
APIGEE_HOST = f"{APIGEE_ORG}-{APIGEE_ENV}.apigee.net"
endpoint_url = f"https://{APIGEE_HOST}/v1/chat/completions"

payload_gemini = {
    "model": "gemini-1.5-flash",
    "messages": [
        {"role": "user", "content": "Explain quantum computing in one sentence."}
    ]
}

headers = {
    "Content-Type": "application/json",
    "X-Api-Key": os.getenv("APIGEE_API_KEY", "test-api-key")
}

print(f"Sending Gemini request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_gemini, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)


In [ ]:
# @title Test 2: OpenAI Target (GPT-4o)
payload_openai = {
    "model": "gpt-4o",
    "messages": [
        {"role": "user", "content": "What is the capital of France?"}
    ]
}

print(f"Sending OpenAI request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_openai, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)


In [ ]:
# @title Test 3: Anthropic Target (Claude 3.5 Sonnet)
payload_claude = {
    "model": "claude-3-5-sonnet-20240620",
    "messages": [
        {"role": "user", "content": "List 3 key benefits of an enterprise API gateway."}
    ],
    "max_tokens": 150
}

print(f"Sending Anthropic request to {endpoint_url}...")
try:
    response = requests.post(endpoint_url, headers=headers, json=payload_claude, timeout=30)
    print(f"Status Code: {response.status_code}")
    print("Response:", json.dumps(response.json(), indent=2))
except Exception as e:
    print("Execution note:", e)
